In [50]:
from dfft.utils.files import get_path
import os
import pandas as pd
import numpy as np
import re
import json

In [2]:
def extract_time(d):
    m = re.match(r"(\d{8}_\d{6})", os.path.basename(d))
    return m.group(1) if m else ""

def get_lastest_exp(base_dir):
    exp_dirs = [os.path.join(base_dir, d) for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    exp_dirs = sorted(exp_dirs, key=extract_time)
    return exp_dirs[-1]

In [47]:
names = ['moment-small', 'moment-base', 'moment-large']
model_configs=["-zeroshot", "-finetuned", "-lora"]

In [58]:
all=[]
for name in names:
    for c in model_configs:
        model = name + c
        base_dir = get_path(f"results\\experiments\\{model}\\NEMA")
        exp_dir = get_lastest_exp(base_dir)
        #Read raw inference output
        df = pd.read_parquet(exp_dir + "/results.parquet")
        #Compute regret
        result = df.groupby('split')[['true_proft', 'real_profit']].sum()
        result['model'] = model
        result['regret'] = (result['true_proft'] - result['real_profit']) / result['true_proft']
        all.append(result.reset_index())

regret = pd.concat(all, axis= 0)
#Reshape data
regret = regret[['model', 'split', 'regret']]
regret = regret.pivot(index='model', columns='split', values='regret')
regret = regret.sort_values('val')
print(regret)

split                       test     train       val
model                                               
moment-large-zeroshot   0.955504  1.093350  0.974166
moment-small-lora       1.043789  0.412840  1.023195
moment-base-lora        1.017445  0.425369  1.037453
moment-small-finetuned  1.072708  0.424758  1.038263
moment-large-finetuned  1.038126  0.428834  1.054825
moment-large-lora       1.065691  0.453304  1.054931
moment-small-zeroshot   1.044120  1.504422  1.058501
moment-base-finetuned   1.074085  0.396233  1.086221
moment-base-zeroshot    1.182888  1.204992  1.225036


In [57]:
all=[]
for name in names:
    for c in model_configs:
        model = name + c
        base_dir = get_path(f"results\\experiments\\{model}\\NEMA")
        exp_dir = get_lastest_exp(base_dir)
        with open(exp_dir + "/results.json", 'r') as f:
             data = json.load(f)
        df = pd.DataFrame([model], columns=["model"])
        df["test_mse"] = data['test_results']['mse']
        df["val_mse"] = data['final_val_results']['mse']
        df["train_mse"] = data['final_train_results']['mse']
        all.append(df)

mse  = pd.concat(all, axis= 0)
print(mse)

                    model     test_mse      val_mse    train_mse
0   moment-small-zeroshot  2249.966408  1583.760654  1422.931051
0  moment-small-finetuned  1726.577932  1365.430719   791.311170
0       moment-small-lora  1747.454704  1361.555811   771.505595
0    moment-base-zeroshot  2258.879695  1566.110933  1384.516358
0   moment-base-finetuned  1781.987679  1368.743915   717.659777
0        moment-base-lora  1745.091426  1359.462069   768.552798
0   moment-large-zeroshot  2198.123922  1530.064325  1382.598652
0  moment-large-finetuned  1699.624718  1361.238795   828.298552
0       moment-large-lora  1723.962507  1369.072999   871.459472


In [60]:
metrics = pd.merge(mse, regret, on='model', how='inner')
print(metrics.sort_values('val'))

                    model     test_mse      val_mse    train_mse      test  \
6   moment-large-zeroshot  2198.123922  1530.064325  1382.598652  0.955504   
2       moment-small-lora  1747.454704  1361.555811   771.505595  1.043789   
5        moment-base-lora  1745.091426  1359.462069   768.552798  1.017445   
1  moment-small-finetuned  1726.577932  1365.430719   791.311170  1.072708   
7  moment-large-finetuned  1699.624718  1361.238795   828.298552  1.038126   
8       moment-large-lora  1723.962507  1369.072999   871.459472  1.065691   
0   moment-small-zeroshot  2249.966408  1583.760654  1422.931051  1.044120   
4   moment-base-finetuned  1781.987679  1368.743915   717.659777  1.074085   
3    moment-base-zeroshot  2258.879695  1566.110933  1384.516358  1.182888   

      train       val  
6  1.093350  0.974166  
2  0.412840  1.023195  
5  0.425369  1.037453  
1  0.424758  1.038263  
7  0.428834  1.054825  
8  0.453304  1.054931  
0  1.504422  1.058501  
4  0.396233  1.086221  
3  